# How big is the KEV residual? Normalization matters

Aidi's comment: *the total tendency almost cancels in a spatial integral, so using it as the denominator is
not a fair comparison. On the test date the residual is 20% of $\overline{\int \mathrm{IVKETEND}}$ but
only 1% of $\overline{\int |\mathrm{IVKETEND}|}$ (the L1 norm, which cannot cancel).*

This notebook reproduces that on his test case, 2017-07-12, over his box (170–120°W, 25–55°N), and puts it
next to the other ways of measuring the residual, both **box-integrated** and **map-resolved**.

Two residuals are compared:

| | definition | what it tests |
|---|---|---|
| **Aidi's** ("Residual of IKEV budget" in `budget_box_IVKE1.py`) | tend − model dynamics − subgrid_momentum − net_vapor_source − analysis | MERRA-2's own tendencies: model dynamics = `kev_model_dynamics_momentum` + `kev_model_dynamics_vapor` (IDOKE + IDOV). No boundary term. |
| **offline** (`budget.groups`, the website budget) | tend − hflux − vflux − pe_conversion − subgrid_momentum − net_vapor_source − analysis − boundary | our dynamics, computed offline from the 3-hourly 0.5° output |

**Running it.** The vke-budgets kernel provides `VKE_MERRA2_ROOT_2017` (MERRA-2 for 2017-07-12/13),
`VKE_JULY_OUT` (where the day's budget and the figures, as PNG, are written) and, optionally, `VKE_AIDI_OUTPUT` (Aidi's output, for a
cross-check against his own box budget).

In [ ]:
import os

import cartopy.crs as ccrs
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from vke_budgets import budget, io, pipeline
from vke_budgets.sources import merra2

MERRA2_ROOT = os.environ["VKE_MERRA2_ROOT_2017"]
OUT = os.environ["VKE_JULY_OUT"]
AIDI_OUTPUT = os.environ.get("VKE_AIDI_OUTPUT")

BOX = {"lon": (190.0, 240.0), "lat": (25.0, 55.0)}      # Aidi's box, 170-120 W, 25-55 N
MAP = {"lon": (160.0, 260.0), "lat": (15.0, 65.0)}      # context for the maps
DAY = 86400.0
BLUE, ORANGE = "#2a78d6", "#eb6834"                      # validated categorical pair

# Figures are also saved here as PNG, ready to send.
FIGURES = os.path.join(OUT, "figures")
os.makedirs(FIGURES, exist_ok=True)


def save(fig, name):
    fig.savefig(os.path.join(FIGURES, name), dpi=200, bbox_inches="tight")

## 1. The budget of 2017-07-12

Computed with the package's default geometry and MERRA-2's own dynamics tendencies (`dyn_crosscheck`), written
once to `VKE_JULY_OUT` (about 1.5 minutes).

In [ ]:
options = budget.BudgetOptions(quantities=("KEV",), dyn_crosscheck=True)
pipeline.run("2017-07-12", "2017-07-12", MERRA2_ROOT, OUT, options, n_workers=3)
ds = io.open_budget(os.path.join(OUT, "budget_*.nc")).load()
g = budget.groups(ds, "KEV")

model_dynamics = ds["kev_model_dynamics_momentum"] + ds["kev_model_dynamics_vapor"]
physics = g["subgrid_momentum"] + g["net_vapor_source"] + g["analysis"]
residuals = {
    "Aidi's residual (model dynamics)": g["tend"] - model_dynamics - physics,
    "offline residual (budget.groups)": g["residual"],
}
tend = g["tend"]

## Helpers

`area_integral` weights by cos(latitude), as Aidi's `integ_xy_s` does (the constant grid spacing cancels in
every ratio). MERRA-2 longitudes run from −180 to 180; `region` cuts out a 0–360° range.

In [ ]:
def region(da, bounds):
    da = da.assign_coords(lon=da["lon"] % 360).sortby("lon")
    return da.sel(lon=slice(*bounds["lon"]), lat=slice(*bounds["lat"]))


def area_integral(da):
    """Integral over the latitude-longitude domain of da, weighted by cos(lat)."""
    return (da * np.cos(np.deg2rad(da["lat"]))).sum(("lat", "lon"))


def area_rms(da):
    weights = np.cos(np.deg2rad(da["lat"])) * xr.ones_like(da)
    return float(np.sqrt(area_integral(da**2).sum() / area_integral(weights).sum()))


def cancelled(da):
    """Share of the field that cancels in the box integral, 1 - |mean int x| / mean int |x|."""
    return 1 - abs(float(area_integral(da).mean())) / float(area_integral(abs(da)).mean())


def normalizations(r, t, kev):
    """Residual r against tendency t over a region, five ways (all time means over the windows)."""
    groups_box = [area_integral(region(g[n], BOX)) for n in g.data_vars if n not in ("tend", "residual")]
    size = sum(abs(x) for x in groups_box)          # size of the box budget, per window
    return {
        "signed / signed (Aidi's 20%)":
            float((area_integral(r) / kev).mean() / (area_integral(t) / kev).mean()),
        "signed / L1 (Aidi's 1%)": float(area_integral(r).mean() / area_integral(abs(t)).mean()),
        "L1 / L1": float(area_integral(abs(r)).mean() / area_integral(abs(t)).mean()),
        "RMS / RMS": area_rms(r) / area_rms(t),
        "|box resid| / sum |box groups|": float((abs(area_integral(r)) / size).mean()),
    }

## 2. Box-integrated: the budget over Aidi's box

Time means over the 8 windows of the box integrals, per unit box KEV and per day: Aidi's units, so the
numbers can be read against his `budget_box_20170712_timemean.csv`. The net tendency is a small remainder of
large terms of opposite sign.

In [ ]:
kev_box = area_integral(region(ds["kev_mean"], BOX))


def box_per_day(da):
    return float((area_integral(region(da, BOX)) / kev_box).mean() * DAY)


rows = {name: box_per_day(g[name]) for name in g.data_vars if name not in ("residual",)}
rows.update({name: box_per_day(r) for name, r in residuals.items()})
labels = list(rows)
colors = [ORANGE if "residual" in name else BLUE for name in labels]

fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.barh(labels, [rows[n] for n in labels], color=colors, height=0.6)
ax.bar_label(bars, fmt="%+.3f", padding=3, fontsize=8, color="0.25")
ax.axvline(0, color="0.6", linewidth=0.8)
ax.invert_yaxis()
ax.set_xlabel("time-mean box integral / box KEV (day$^{-1}$)")
ax.set_title("KEV budget over 170–120°W, 25–55°N, 2017-07-12", fontsize=10)
ax.legend(handles=[mpatches.Patch(color=BLUE, label="budget groups and tend"),
                   mpatches.Patch(color=ORANGE, label="residuals")], frameon=False, fontsize=8,
          loc="lower left")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
ax.margins(x=0.15)
save(fig, "1_box_budget.png")

Cross-check against Aidi's own box budget (his code, his output files), if `VKE_AIDI_OUTPUT` is set.

In [ ]:
if AIDI_OUTPUT:
    aidi = pd.read_csv(os.path.join(AIDI_OUTPUT, "budget_box_20170712_timemean.csv"), index_col=0)
    his = aidi.loc["Residual of IKEV budget", "Integ_lev0_perday"] / aidi.loc["IKEV tendency", "Integ_lev0_perday"]
    ours = rows["Aidi's residual (model dynamics)"] / rows["tend"]
    print(f"Aidi's residual / tend, box, time mean:  his files {his:.3f}   this package {ours:.3f}")

## 3. Box-integrated: five ways to normalize

Each ratio is a time mean over the 8 windows, in the box. *Signed* is the plain integral of the field, *L1*
the integral of its absolute value, *RMS* the area-weighted root mean square over points and windows. The last
row compares the box-integrated residual with the size of the box budget itself (the sum of the absolute box
integrals of the groups), not with the near-cancelling net tendency. Signed ratios are shown as absolute
values.

In [ ]:
t_box = region(tend, BOX)
table = pd.DataFrame({name: normalizations(region(r, BOX), t_box, kev_box) for name, r in residuals.items()})
display(table.style.format("{:+.3f}"))

fig, ax = plt.subplots(figsize=(7.5, 3.8))
y = np.arange(len(table))
for k, (name, color) in enumerate(zip(table.columns, (BLUE, ORANGE))):
    bars = ax.barh(y + (k - 0.5) * 0.38, table[name].abs(), height=0.36, color=color, label=name)
    ax.bar_label(bars, labels=[f"{v:.1%}" for v in table[name].abs()], padding=3, fontsize=8,
                 color="0.25")
ax.set_yticks(y, table.index)
ax.invert_yaxis()
ax.set_xlabel("residual / tendency (absolute value)")
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("The same residuals, five normalizations (box, 2017-07-12)", fontsize=10)
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.margins(x=0.12)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
save(fig, "2_normalizations.png")

Why the signed ratios mislead in *both* directions: the box integral cancels most of the tendency (so
signed/signed inflates any residual), but it also cancels most of the residual, which changes sign from grid
point to grid point (so signed/L1 deflates it). A fair pointwise ratio treats numerator and denominator alike
(L1/L1, RMS/RMS).

In [ ]:
shares = {"tend": cancelled(t_box), **{name: cancelled(region(r, BOX)) for name, r in residuals.items()}}

fig, ax = plt.subplots(figsize=(6.5, 2.3))
bars = ax.barh(list(shares), list(shares.values()), color=BLUE, height=0.55)
ax.bar_label(bars, labels=[f"{v:.0%}" for v in shares.values()], padding=3, fontsize=8, color="0.25")
ax.set_xlim(0, 1.08)
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.invert_yaxis()
ax.set_xlabel("share cancelled in the box integral, 1 − |∫x| / ∫|x|")
ax.set_title("How much of each field cancels in the box integral", fontsize=10)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
save(fig, "3_cancellation.png")

## 4. Map-resolved

Where are the residuals? First one window, on one shared color scale; then the time mean of the absolute
values over the 8 windows (the pointwise L1 picture). Aidi's box is outlined.

In [ ]:
def maps(fields, title, cmap, vmin, vmax, units="kg s$^{-3}$"):
    fig, axes = plt.subplots(1, len(fields), figsize=(4.3 * len(fields), 2.6), squeeze=False,
                             layout="constrained",
                             subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})
    for ax, (name, f) in zip(axes.flat, fields.items()):
        f = region(f, MAP)
        mesh = ax.pcolormesh(f["lon"], f["lat"], f, cmap=cmap, vmin=vmin, vmax=vmax,
                             transform=ccrs.PlateCarree(), shading="auto")
        ax.coastlines(linewidth=0.5, color="0.35")
        ax.add_patch(mpatches.Rectangle((BOX["lon"][0], BOX["lat"][0]), np.diff(BOX["lon"])[0],
                                        np.diff(BOX["lat"])[0], fill=False, edgecolor="0.15",
                                        linewidth=1.0, transform=ccrs.PlateCarree()))
        ax.set_extent([*MAP["lon"], *MAP["lat"]], crs=ccrs.PlateCarree())
        ax.set_title(name, fontsize=9)
    fig.colorbar(mesh, ax=axes, shrink=0.9, label=units)
    fig.suptitle(title, fontsize=10)
    return fig


frame = 4
one = {"tend": tend.isel(time=frame), **{n: r.isel(time=frame) for n, r in residuals.items()}}
vmax = float(np.nanpercentile(np.abs(region(one["tend"], MAP)), 98))
fig = maps(one, f"KEV budget, window centred {str(ds['time'].values[frame])[:16]}", "RdBu_r", -vmax, vmax)
save(fig, "4_maps_one_window.png")

In [ ]:
mean_abs = {"|tend|": abs(tend).mean("time"), **{f"|{n}|": abs(r).mean("time") for n, r in residuals.items()}}
vmax = float(np.nanpercentile(region(mean_abs["|tend|"], MAP), 98))
fig = maps(mean_abs, "Time mean of the absolute values, 2017-07-12 (8 windows)", "Blues", 0, vmax)
save(fig, "5_maps_mean_absolute.png")
print("figures saved in", FIGURES)

## Summary

Printed from the numbers above:

In [ ]:
a, o = table.columns
print(f"""In the box (170-120 W, 25-55 N), 2017-07-12:
- The net tendency cancels {shares['tend']:.0%} in the box integral, so dividing by it inflates any residual:
  Aidi's residual is {table.loc['signed / signed (Aidi\'s 20%)', a]:+.1%} of it (his 20%).
- Against the L1 tendency it is {table.loc['signed / L1 (Aidi\'s 1%)', a]:+.1%} (his 1%), but its own signed
  integral cancels {shares[a]:.0%}, so that ratio understates it.
- Treating both alike: Aidi's residual is {table.loc['L1 / L1', a]:.1%} (L1/L1) or {table.loc['RMS / RMS', a]:.1%} (RMS/RMS)
  of the tendency; the offline residual is {table.loc['L1 / L1', o]:.1%} or {table.loc['RMS / RMS', o]:.1%}.
- Box-integrated, against the size of the box budget: {table.loc['|box resid| / sum |box groups|', a]:.1%} (Aidi's), {table.loc['|box resid| / sum |box groups|', o]:.1%} (offline).
So MERRA-2's own tendencies close the budget to a few percent everywhere, while the offline dynamics leave a
large grid-scale residual point by point that mostly cancels in box integrals.""")